# Risk Management — Variance Forecasts and Factor Risk Limits
## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Estimate and forecast portfolio volatility** using realized variance and AR(1)
2. **Understand VIX** as the market's implied vol — compare it to historical realized vol
3. **Apply factor risk limits** — how pod shops constrain exposure to individual factors
4. **Compute Value-at-Risk (VaR) and Expected Shortfall** for a portfolio
5. **Audit AI-generated risk metrics** — frequency consistency, tail-distribution assumptions

## 📋 TOC
1. [Setup](#setup)  2. [Variance Forecasting Basics](#variance)
3. [Pitfall Checklist](#pitfalls)  4. [Realized Variance and AR(1)](#realized)
5. [VIX and Implied Vol](#vix)  6. [Factor Risk Limits](#limits)
7. [VaR and Expected Shortfall](#var)  8. [🎯 Challenge: Risk-Budget a Portfolio](#challenge)
9. [Submission](#submit)  10. [Key Takeaways](#takeaways)

---
## 🛠️ Setup <a id="setup"></a>

In [ ]:
#@title Setup
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize']=[10,5]; plt.rcParams['font.size']=11
import warnings; warnings.filterwarnings('ignore')
print("✅ Loaded")

---
## Variance Forecasting Basics <a id="variance"></a>

We deferred this from the Estimation lecture (L5). Now we tackle it.

**The good news about variances:** unlike means, variances are *forecastable*.
The empirical regularity: high-vol months are followed by high-vol months,
and low-vol by low-vol. This is **volatility clustering**.

Robert Engle won the 2003 Nobel for ARCH/GARCH models that exploit this.
You don't need ARCH/GARCH to capture most of it — a simple AR(1) on
realized variance does ~80% of the work.

---
## 🛡️ Pitfall Checklist <a id="pitfalls"></a>

| | Pitfall | What goes wrong | 🔍 How to detect |
|---|---------|-----------------|-------------------|
| 1 | **Confusing variance with vol** | Annualization: σ × √N, σ² × N | Compute both ways, sanity-check magnitudes |
| 2 | **Overlapping rolling windows** | 252-day rolling vol uses 251 shared days from yesterday | Use non-overlapping windows for hypothesis tests |
| 3 | **Tail assumption: normality** | VaR from normal under-predicts tail losses by 2-3x | Compare actual 1st percentile to normal model |
| 4 | **Stale prices in idiosyncratic vol** | When you use beta from one period and residuals from another | Recompute beta inside the window |
| 5 | **Factor risk added linearly** | Factor exposures don't add — they correlate | Use the covariance matrix, not absolute sums |

---
## Realized Variance and AR(1) <a id="realized"></a>

For each month, **realized variance** = sum of squared daily returns within that month:

$$RV_t = \sum_{d \in \text{month } t} r_d^2$$

Then forecast next month's RV from last month's:

$$RV_{t+1} = a + b \cdot RV_t + \epsilon_{t+1}$$

Typically $b \approx 0.5-0.8$ — strong persistence.

In [ ]:
# Build realized variance for the market, fit AR(1)
from pandas_datareader.data import DataReader
ff = DataReader("F-F_Research_Data_Factors_daily", "famafrench", start="1990-01-01")[0] / 100
mkt_d = ff['Mkt-RF']

# Monthly RV
rv = mkt_d.groupby(mkt_d.index.to_period('M')).apply(lambda x: (x**2).sum())
rv.index = rv.index.to_timestamp(how='end').normalize()
rv.name = 'RV'

# AR(1) regression
X = sm.add_constant(rv.shift(1).dropna())
y = rv.loc[X.index]
ar1 = sm.OLS(y, X).fit()
print(ar1.summary().tables[1])
print(f"\nR² of AR(1) on monthly RV: {ar1.rsquared:.3f}")

---
## VIX and Implied Vol <a id="vix"></a>

The **VIX** is the market's **risk-neutral** expectation of S&P 500 vol
over the next 30 days. It's computed from option prices, and you can read
it off Bloomberg in real time.

**VIX vs realized vol:**
- VIX tends to be **higher** than subsequent realized vol on average — this is the **variance risk premium**, which is itself tradable
- VIX **spikes BEFORE** realized vol does, because options price in expected future shocks
- VIX is your best forward-looking variance forecast for short horizons

---
## Factor Risk Limits <a id="limits"></a>

Pod shops enforce **factor risk limits** to keep individual managers from
inadvertently making a market or sector bet:

$$\text{Vol contribution from factor } k = |\beta_k| \cdot \sigma_k$$

Each manager has caps like:
- Total portfolio vol ≤ 8%
- Factor (MKT, sector) vol contribution ≤ 1%
- Single-name vol contribution ≤ 0.5%

These constraints force the manager to **hedge any factor exposure they
didn't intend** — which is the whole point of factor-neutral trading.

---
## VaR and Expected Shortfall <a id="var"></a>

**Value-at-Risk (VaR)** at α: the loss level such that P(loss > VaR) = α.

Common: 1-day 95% VaR. If your portfolio's 1-day VaR is $1M, then 5% of
days you'll lose more than $1M.

**Expected Shortfall (ES)**: the average loss *conditional on* exceeding VaR:

$$ES_\alpha = E[L \mid L > VaR_\alpha]$$

ES is **always larger** than VaR — it measures the *severity* of tail losses,
not just their frequency. Banks and hedge funds increasingly use ES instead
of VaR because VaR understates tail risk.

In [ ]:
# Compute 1-day 95% VaR for SPY from history
spy = mkt_d   # use Mkt-RF as proxy for SPY excess
var_95 = spy.quantile(0.05)
es_95  = spy[spy <= var_95].mean()

print(f"Daily VaR (95%): {var_95:.2%}    (5% of days you lose more than this)")
print(f"Expected Shortfall (95%): {es_95:.2%}    (average loss on those bad days)")
print(f"\nNote ES is ~{abs(es_95/var_95):.1f}× larger than VaR — tail asymmetry.")

---
## 🎯 Challenge: Risk-Budget a Portfolio <a id="challenge"></a>

> **Setup.** Your portfolio:
> - $50M long the market (β=1.0)
> - $20M long a value strategy (β=0.3 to MKT, σ_idio = 8%)
> - Portfolio vol target: 12% annualized

### Q1 — Total portfolio vol assuming independence

Assume market vol = 18%, value-strategy idiosyncratic vol = 8%, uncorrelated.

> **📌 Required:**
> ```python
> w_market   = 0.50    # $50M / $100M total NAV (assume $100M NAV for simplicity)
> w_value    = 0.20
> sig_market = 0.18
> sig_value_idio = 0.08
> portfolio_vol = ____   # sqrt( (w_m * sig_m)^2 + (w_v * sig_v_idio)^2 ), uncorrelated
> ```

In [ ]:
w_market = 0.50
w_value  = 0.20
sig_market     = 0.18
sig_value_idio = 0.08

portfolio_vol = ____
print(f"Portfolio vol (uncorrelated approx): {portfolio_vol:.2%}")

### Q2 — VaR

Assuming normal returns, 1-day 95% VaR ≈ $NAV × portfolio_vol / √252 × 1.65.

> **📌 Required:**
> ```python
> nav        = 100_000_000
> daily_vol  = portfolio_vol / np.sqrt(252)
> var_95_dollars = ____   # nav * daily_vol * 1.65
> ```

In [ ]:
nav = 100_000_000

daily_vol = portfolio_vol / np.sqrt(252)
var_95_dollars = ____
print(f"Daily 95% VaR: ${var_95_dollars:,.0f}")

### Q3 — Are you within vol budget?

> **📌 Required:**
> ```python
> vol_target = 0.12
> within_budget = ____    # 1.0 if portfolio_vol <= vol_target, else 0.0
> ```

In [ ]:
vol_target = 0.12

within_budget = ____
print(f"Within {vol_target:.0%} budget? {bool(within_budget)}")

### Q4 — Memo

Max 5 sentences. State (i) portfolio vol, (ii) daily VaR, (iii) whether
within budget and (if not) what you would scale back.

In [ ]:
MEMO = """Write your memo here."""
print(MEMO)

---
## 📤 Submission <a id="submit"></a>

In [ ]:
# === 📤 SUBMISSION CELL ===
import json, base64, hashlib, datetime as dt
required = ["portfolio_vol", "var_95_dollars", "within_budget", "MEMO"]
missing = [v for v in required if v not in dir()]
if missing: raise NameError(f"\n❌ Missing: {missing}")
payload = {"assignment": "RiskManagement_AI",
    "ts": dt.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip()}
blob = json.dumps(payload, sort_keys=True)
token = f"UG54::{hashlib.sha256(blob.encode()).hexdigest()[:8]}::{base64.b64encode(blob.encode()).decode()}"
print("="*72); print(token); print("="*72)

---
## 🧠 Key Takeaways <a id="takeaways"></a>
1. **Variance is forecastable.** A simple AR(1) on realized variance captures most of the persistence.
2. **VIX is forward-looking** and tends to overpredict realized vol — the variance risk premium.
3. **Factor risk limits** constrain managers to take only the risks they intended.
4. **VaR is incomplete.** Expected Shortfall captures tail severity.
5. **AI computes the numbers. You interpret them in context.**